# ⚗️ Notebook 8 — Scenario: External Customer Product Compatibility

This notebook runs the **product compatibility** scenario — the highest-governance flow.

## Governance demonstrated
- **DISCLAIMER_GATE**: compatibility queries are blocked until the customer acknowledges the disclaimer
- **Multi-agent bundle**: product-intelligence + compatibility run in combination, aligner validates
- **Confidence threshold**: if confidence < 0.90 the orchestrator refuses to give a recommendation
- **Safety rationale**: the compatibility verdict always includes evidence-based safety reasoning

## Scenarios
1. Compatibility query without disclaimer → blocked
2. Disclaimer accepted → compatible pair (Clean Pro + Odor Shield)
3. Disclaimer accepted → incompatible pair (Clean Pro + Flea Guard)
4. Sample request → authenticated customer flow

In [ ]:
import sys, json, pathlib, uuid, subprocess
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "shared" / "utils.py").exists() and (candidate / "workshop" / "product-finder").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing shared/utils.py and workshop/product-finder")

repo_root = find_repo_root(pathlib.Path.cwd())
sys.path.insert(0, str(repo_root / "shared"))
import utils  # type: ignore

def run(cmd: str, ok: str = "", fail: str = ""):
    return utils.run(cmd, ok, fail)

def azd_get(key: str) -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Missing azd env value [{key}]: {(p.stderr or p.stdout).strip()}")
    return (p.stdout or "").strip()

def azd_get_optional(key: str, default: str = "") -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        return default
    return (p.stdout or "").strip() or default

ORCHESTRATOR_NAME = azd_get_optional("PF_ORCHESTRATOR_NAME", "pf-orchestrator")
FOUNDRY_EP = azd_get_optional("FOUNDRY_PROJECT_ENDPOINT", "")
if not FOUNDRY_EP:
    acct = azd_get_optional("SPOKE_AI_FOUNDRY_ACCOUNT_NAME", "")
    project = azd_get_optional("SPOKE_AI_FOUNDRY_PROJECT_NAME", "")
    if acct and project:
        FOUNDRY_EP = f"https://{acct}.services.ai.azure.com/api/projects/{project}"
if not FOUNDRY_EP:
    raise RuntimeError("Missing FOUNDRY_PROJECT_ENDPOINT. Run Notebook 6 first.")

project_client = AIProjectClient(endpoint=FOUNDRY_EP, credential=DefaultAzureCredential(), allow_preview=True)
oc = project_client.get_openai_client(agent_name=ORCHESTRATOR_NAME)
utils.print_ok(f"Scenario client ready for orchestrator: {ORCHESTRATOR_NAME}")

def gov_msg(persona: str, user_text: str, disclaimer_accepted: bool = True) -> str:
    return f"""[GOVERNANCE CONTEXT]
persona: {persona}
disclaimer_accepted: {'true' if disclaimer_accepted else 'false'}

{user_text}"""

def show_bundle(title: str, response_text: str, expected_agents=None, checks=None):
    if title:
        print("\n" + "=" * 70)
        print(title)
        print("=" * 70)
    try:
        obj = json.loads(response_text)
        print(json.dumps(obj, indent=2))
    except Exception:
        obj = {"_raw": response_text}
        print(response_text)

    agents_used = [str(a).lower() for a in obj.get("agents_used", [])] if isinstance(obj, dict) else []

    if expected_agents:
        for expected in expected_agents:
            ok = any(expected.lower() in a for a in agents_used)
            if ok:
                utils.print_ok(f"Expected agent present: {expected}")
            else:
                utils.print_warning(f"Expected agent missing: {expected}")

    if checks:
        for label, ok in checks:
            if ok:
                utils.print_ok(label)
            else:
                utils.print_warning(label)

    return obj

### 🔒 Test 1 — Compatibility query WITHOUT disclaimer
The governance layer must block this and return a disclaimer gate response.

In [ ]:
query_compat = 'Can I combine SynPet Clean Pro with SynPet Flea Guard on my healthy 6-year-old golden retriever?'
utils.print_info(f'Query: "{query_compat}"  disclaimer_accepted=False')

resp1 = oc.responses.create(
    input=gov_msg('external_customer', query_compat, disclaimer_accepted=False),
    metadata={'conversation_id': str(uuid.uuid4())},
)
b1 = show_bundle('COMPATIBILITY — no disclaimer (must be BLOCKED)', resp1.output_text)

raw1 = resp1.output_text.lower()
final1 = str(b1.get('final_answer', resp1.output_text)).lower()

# Hard gate expectation: disclaimer_required must be explicit and no compatibility verdict should be given yet.
gate_activated = b1.get('disclaimer_required') is True
compatibility_answer_given = any(
    kw in final1
    for kw in [
        'compatible',
        'incompatible',
        'safe to combine',
        'can combine',
        'you can combine',
        'combining',
        'recommended combination',
    ]
)

show_bundle('', resp1.output_text, checks=[
    ('Disclaimer gate activated (disclaimer_required=True)', gate_activated),
    ('No compatibility answer given yet', not compatibility_answer_given),
])

### ✅ Test 2 — Compatible pair WITH disclaimer accepted
`SynPet Clean Pro + SynPet Odor Shield` — verdict: **compatible**

In [ ]:
query_ok = 'Can I use SynPet Clean Pro together with SynPet Odor Shield on my healthy 6-year-old golden retriever?'
utils.print_info(f'Query: "{query_ok}"  disclaimer_accepted=True')

resp2 = oc.responses.create(
    input=gov_msg('external_customer', query_ok, disclaimer_accepted=True),
    metadata={'conversation_id': str(uuid.uuid4())},
)
b2 = show_bundle('COMPATIBLE PAIR — expect compatible verdict + high confidence', resp2.output_text)

ans2 = b2.get('final_answer', resp2.output_text).lower()
agents_used = [str(a).lower() for a in b2.get('agents_used', [])]
show_bundle('', resp2.output_text, checks=[
    ('Routing intent is compatibility', b2.get('routing_decision', {}).get('intent') == 'compatibility'),
    ('Multi-agent bundle includes compatibility agent', any('compat' in a for a in agents_used)),
    ('Compatible verdict in answer', 'compatible' in ans2),
    ('Confidence above 0.85', float(b2.get('confidence', 0)) > 0.85),
    ('Disclaimer notice included', bool(b2.get('governance_notices'))),
    ('Disclaimer required reflects metadata policy', b2.get('disclaimer_required') is True),
])

### ❌ Test 3 — Incompatible pair WITH disclaimer accepted
`SynPet Clean Pro + SynPet Flea Guard` — verdict: **incompatible**, safety warning expected.

In [ ]:
query_bad = 'Can I mix SynPet Clean Pro with SynPet Flea Guard on my healthy 6-year-old golden retriever?'
utils.print_info(f'Query: "{query_bad}"  disclaimer_accepted=True')

resp3 = oc.responses.create(
    input=gov_msg('external_customer', query_bad, disclaimer_accepted=True),
    metadata={'conversation_id': str(uuid.uuid4())},
)
b3 = show_bundle('INCOMPATIBLE PAIR — expect safety warning', resp3.output_text)

ans3 = b3.get('final_answer', resp3.output_text).lower()
show_bundle('', resp3.output_text, checks=[
    ('Routing intent is compatibility', b3.get('routing_decision', {}).get('intent') == 'compatibility'),
    ('Incompatible verdict or safety warning in answer',
     any(kw in ans3 for kw in ['incompatible', 'do not', 'cannot', 'unsafe', 'caution', 'warning'])),
    ('Confidence-based evidence cited', b3.get('confidence') is not None),
    ('Disclaimer required reflects metadata policy', b3.get('disclaimer_required') is True),
])

### 📦 Test 4 — Sample request (authenticated external customer)

In [ ]:
query_sample = 'I would like to request a sample of SynPet Gentle Care for my healthy 6 year old dog'
utils.print_info(f'Query: "{query_sample}"  persona: external_customer')

resp4 = oc.responses.create(
    input=gov_msg('external_customer', query_sample),
    metadata={'conversation_id': str(uuid.uuid4())},
)
b4 = show_bundle('SAMPLE REQUEST — expect confirmation', resp4.output_text)

ans4 = b4.get('final_answer', resp4.output_text).lower()
show_bundle('', resp4.output_text, checks=[
    ('Sample confirmation or request details in answer',
     any(kw in ans4 for kw in ['sample', 'request', 'confirmation', 'sr-', 'deliver', 'dispatch'])),
])

print()
utils.print_ok('✅ Compatibility scenario COMPLETE. Proceed to Notebook 9.')